# Analyse Water Storage: Surface Waterbodies

Read a published waterbody asset, plot its water extent and explore individual API records.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the location

These three fields contain the selected tehsil when downloaded from GeoLibre. Edit them to explore another location, then restart the kernel and run from the top.


In [ ]:
state = "Bihar"
district = "Nalanda"
tehsil = "Hilsa"


## Set your API key

The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and keys. This cell reuses `CORE_STACK_API_KEY` or asks privately and stores it in this kernel’s environment. The request header is `X-API-Key`.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", value.replace("(", "").replace(")", "")).strip("_").lower()
         for key, value in {"state": state, "district": district, "tehsil": tehsil}.items()}
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, "{state}/{district}/{tehsil}/collection.json".format(**place))
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "surface_water_bodies_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
field_notes = pd.DataFrame(columns=["name", "type", "description"])
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(pd.DataFrame([item["properties"]]).reindex(columns=["title", "description", "start_datetime", "end_datetime"]).T)
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Choose a waterbody from the STAC asset

Follow the GeoJSON asset link published in STAC. Choose a `waterbody_id` from its `UID` column. The selected values appear with their original field names and any published descriptions.


In [ ]:
waterbodies = gpd.GeoDataFrame()
waterbody = pd.Series(dtype=object)
if item is not None:
    response = requests.get(urljoin(item_url, item["assets"]["data"]["href"]), timeout=180)
    waterbodies = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
    display(waterbodies[["UID"]].head(10))
    waterbody_id = str(waterbodies.iloc[0]["UID"])
    waterbody = waterbodies.loc[waterbodies["UID"].astype(str) == waterbody_id].iloc[0]
    columns = ["UID", "area_ored", "area_17-18", "k_17-18", "kr_17-18", "krz_17-18"]
    display(waterbody.reindex(columns).to_frame("value").join(field_notes.set_index("name")[["description", "type"]]))


## Water extent through time

The annual `area_YY-YY` fields are hectares. The seasonal `k_`, `kr_` and `krz_` fields are percentages of `area_ored`, also in hectares. Keep the raw values and add a calculated hectare column for plotting.


In [ ]:
if not waterbody.empty:
    prefix = "area"  # Try "k", "kr" or "krz" for a seasonal view.
    fields = [f"{prefix}_{y%100:02d}-{(y+1)%100:02d}" for y in YEARS]
    extent = pd.to_numeric(waterbody.reindex(fields), errors="coerce").to_frame("value")
    extent = extent.join(field_notes.set_index("name")[["description", "type"]])
    extent["area_in_ha"] = extent["value"] if prefix == "area" else extent["value"] * waterbody["area_ored"] / 100
    display(extent)
    plt.figure(figsize=(9, 3))
    plt.plot(YEARS, extent["area_in_ha"], marker="o", label=prefix)
    plt.ylabel("Water area (ha)")
    plt.title(f"{prefix} · {waterbody_id}")
    plt.grid(alpha=0.2)
    plt.show()


### Try another field

Change `prefix` to compare the seasonal fields without changing the identifier. To inspect annual totals across the tehsil, use `waterbodies[fields].sum(min_count=1)` with `prefix = "area"`, and `waterbodies[fields].count()` to check how many records supply each year. These are mapped areas, not storage volumes.


## Read one waterbody from the API

First obtain the API’s own identifiers with `get_waterbodies_data_by_admin`. Use one of those in `get_waterbody_data`; STAC and API identifiers are not assumed to match. Select a property group and display it as a table.


In [ ]:
response = requests.get(API_URL + "get_waterbodies_data_by_admin/", params=place, headers=api_headers, timeout=180)
inventory = read_json(response)
display(pd.DataFrame({"uid": list(inventory)}).head(10))
properties = {}
if inventory:
    api_waterbody_id = next(iter(inventory))
    response = requests.get(API_URL + "get_waterbody_data/", params={**place, "uid": api_waterbody_id}, headers=api_headers, timeout=180)
    api_waterbody = read_json(response)[api_waterbody_id]
    display(pd.DataFrame({"property_group": list(api_waterbody)}))
    property_group = "zoi_properties"
    properties = api_waterbody.get(property_group, {})
    display(pd.DataFrame({"field": list(properties)}))
    columns = list(properties)[:8]
    display(pd.Series(properties, dtype=object).reindex(columns).to_frame("value"))


## Read a dated property

Some property groups store a time series inside a JSON string. This example reads `NDVI_2017` into a DataFrame before plotting it. Change `field` to another dated field from the list above.


In [ ]:
field = "NDVI_2017"
dated_values = properties.get(field)
if dated_values:
    dated_values = json.loads(dated_values, parse_constant=lambda value: None) if isinstance(dated_values, str) else dated_values
    dated = pd.Series(dated_values, name=field).to_frame()
    dated.index = pd.to_datetime(dated.index)
    dated[field] = pd.to_numeric(dated[field], errors="coerce")
    display(dated.head())
    dated.sort_index().plot(figsize=(9, 3), ylabel="NDVI (unitless)", ylim=(-1, 1))
    plt.show()
else:
    print("This property group does not contain the selected dated field. Choose another field from the table above.")
